# DTI Prediction v17

## d_3D dict key 分类（已确认）
- **原子特征** (mean-pool over atoms): atomic_num, chiral_tag, degree, explicit_valence, formal_charge, hybridization, implicit_valence, is_aromatic, total_numHs, mass → 各 1 dim
- **原子3D坐标** atom_pos (n,3) → mean-pool → 3 dim
- **键特征** (mean-pool over bonds): bond_dir, bond_type, is_in_ring, bond_length → 各 1 dim
- **键角/距离统计**: bond_angle (mean→1), Ba_bond_angle (mean→1), Bl_bond_length (mean→1), Ad_atom_dist (n,1) mean→1
- **固定分子指纹** (直接用): morgan_fp(200), maccs_fp(167), daylight_fg_counts(127)
- **跳过** (图索引/字符串): smiles, edges, BondAngleGraph_edges, Ba_node_vi/j/k, Bl_node_i/j, Ad_node_i/j

总 D3_DIM = 10×1 + 3 + 4×1 + 3×1 + 1 + 200 + 167 + 127 = 515

In [1]:
import os, gc, time, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score, confusion_matrix
)
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ================================================================
# Cell 1: 加载数据
# 行号对齐: d_2D[i] / d_3D[i] / t_feat[i] == df.iloc[i] (同一 pair)
# ================================================================
DATA_PATH = '/kaggle/input/datasets/shiyunsg/bond-angle'

d_2D_raw   = np.load(f'{DATA_PATH}/d_feature.npy',    allow_pickle=True)  # (33062, 256)
d_3D_raw   = np.load(f'{DATA_PATH}/d_3D_feature.npy', allow_pickle=True)  # (33062,) object
t_feat_raw = np.load(f'{DATA_PATH}/t_feature.npy',    allow_pickle=True)  # (33062,) object
df         = pd.read_csv(f'{DATA_PATH}/total_cid_unid_csv.csv')           # (33062, 3)

print(f'd_2D   : {d_2D_raw.shape}, {d_2D_raw.dtype}')
print(f'd_3D   : {d_3D_raw.shape}, {d_3D_raw.dtype}')
print(f't_feat : {t_feat_raw.shape}, {t_feat_raw.dtype}')
print(f'df     : {df.shape}, Pos={(df["label"]==1).sum()}, Neg={(df["label"]==0).sum()}')

assert len(d_2D_raw) == len(df) == len(d_3D_raw) == len(t_feat_raw)
print('Row alignment OK.')

D2_DIM = d_2D_raw.shape[1]  # 256
print(f'D2_DIM={D2_DIM}')

d_2D   : (33062, 256), float32
d_3D   : (33062,), object
t_feat : (33062,), object
df     : (33062, 3), Pos=16531, Neg=16531
Row alignment OK.
D2_DIM=256


In [3]:
# ================================================================
# Cell 2: 构建 d_3D feature array
#
# 精确分类每个 key:
#   ATOM_SCALAR : (n_atoms,)      → mean → 1 dim
#   ATOM_VEC    : (n_atoms, k)    → mean over atoms → k dims
#   BOND_SCALAR : (n_bonds,)      → mean → 1 dim
#   FIXED_FP    : (fixed_dim,)    → 直接用
#   SKIP        : 图索引/字符串   → 丢弃
# ================================================================

# 原子级标量特征 (每个 shape=(n_atoms,))
ATOM_SCALAR_KEYS = [
    'atomic_num', 'chiral_tag', 'degree', 'explicit_valence',
    'formal_charge', 'hybridization', 'implicit_valence',
    'is_aromatic', 'total_numHs', 'mass'
]
# 原子级向量特征 (每个 shape=(n_atoms, k))
ATOM_VEC_KEYS = ['atom_pos']  # (n_atoms, 3) → mean → 3 dims

# 键级标量特征 (每个 shape=(n_bonds,))
BOND_SCALAR_KEYS = ['bond_dir', 'bond_type', 'is_in_ring', 'bond_length']

# 键角/距离统计标量
ANGLE_SCALAR_KEYS = ['bond_angle', 'Ba_bond_angle', 'Bl_bond_length']

# Ad_atom_dist: shape=(n, 1) → mean → 1 dim
AD_DIST_KEY = 'Ad_atom_dist'

# 固定维度分子指纹 (直接 concatenate)
FIXED_FP_KEYS = ['morgan_fp', 'maccs_fp', 'daylight_fg_counts']  # 200+167+127=494

# 跳过: smiles(str), edges(索引), BondAngleGraph_edges(索引),
#        Ba_node_vi/j/k(索引), Bl_node_i/j(索引), Ad_node_i/j(索引)

# 计算总维度
D3_ATOM_S  = len(ATOM_SCALAR_KEYS)   # 10
D3_ATOM_V  = 3                        # atom_pos → 3 dims
D3_BOND_S  = len(BOND_SCALAR_KEYS)   # 4
D3_ANGLE_S = len(ANGLE_SCALAR_KEYS)  # 3
D3_AD_DIST = 1
D3_MORGAN  = 200
D3_MACCS   = 167
D3_DFG     = 127
D3_DIM = D3_ATOM_S + D3_ATOM_V + D3_BOND_S + D3_ANGLE_S + D3_AD_DIST + D3_MORGAN + D3_MACCS + D3_DFG
print(f'D3_DIM = {D3_ATOM_S}+{D3_ATOM_V}+{D3_BOND_S}+{D3_ANGLE_S}+{D3_AD_DIST}+{D3_MORGAN}+{D3_MACCS}+{D3_DFG} = {D3_DIM}')


def safe_mean(arr, fallback=0.0):
    """Mean that returns fallback (not NaN) for empty arrays."""
    return float(arr.mean()) if len(arr) > 0 else fallback

def safe_mean_vec(arr, ndim, fallback=0.0):
    """Row-wise mean for 2D arrays; returns zeros for empty arrays."""
    if arr.shape[0] == 0:
        return np.zeros(ndim, dtype=np.float32)
    return arr.mean(axis=0).flatten().astype(np.float32)

def dict_to_vec(d):
    parts = []

    # 1. Atom scalar features — safe mean avoids NaN on empty molecules
    for k in ATOM_SCALAR_KEYS:
        v = d.get(k, None)
        if v is None:
            parts.append(np.zeros(1, dtype=np.float32))
        else:
            arr = np.asarray(v, dtype=np.float32).flatten()
            parts.append(np.array([safe_mean(arr)], dtype=np.float32))

    # 2. atom_pos: (n_atoms, 3) → mean over atoms → 3 dims
    for k in ATOM_VEC_KEYS:
        v = d.get(k, None)
        if v is None:
            parts.append(np.zeros(3, dtype=np.float32))
        else:
            arr = np.asarray(v, dtype=np.float32)
            if arr.ndim == 1:
                arr = arr.reshape(-1, 1)
            parts.append(safe_mean_vec(arr, arr.shape[1] if arr.ndim > 1 else 1))

    # 3. Bond scalar features
    for k in BOND_SCALAR_KEYS:
        v = d.get(k, None)
        if v is None:
            parts.append(np.zeros(1, dtype=np.float32))
        else:
            arr = np.asarray(v, dtype=np.float32).flatten()
            parts.append(np.array([safe_mean(arr)], dtype=np.float32))

    # 4. Angle/distance scalars
    for k in ANGLE_SCALAR_KEYS:
        v = d.get(k, None)
        if v is None:
            parts.append(np.zeros(1, dtype=np.float32))
        else:
            arr = np.asarray(v, dtype=np.float32).flatten()
            parts.append(np.array([safe_mean(arr)], dtype=np.float32))

    # 5. Ad_atom_dist
    v = d.get(AD_DIST_KEY, None)
    if v is None:
        parts.append(np.zeros(1, dtype=np.float32))
    else:
        arr = np.asarray(v, dtype=np.float32).flatten()
        parts.append(np.array([safe_mean(arr)], dtype=np.float32))

    # 6. Fixed fingerprints
    for k, expected_dim in zip(FIXED_FP_KEYS, [D3_MORGAN, D3_MACCS, D3_DFG]):
        v = d.get(k, None)
        if v is None:
            parts.append(np.zeros(expected_dim, dtype=np.float32))
        else:
            arr = np.asarray(v, dtype=np.float32).flatten()
            if len(arr) != expected_dim:
                tmp = np.zeros(expected_dim, dtype=np.float32)
                tmp[:min(len(arr), expected_dim)] = arr[:expected_dim]
                arr = tmp
            # Replace any residual NaN/inf with 0
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            parts.append(arr)

    result = np.concatenate(parts).astype(np.float32)
    # Final safety: replace any NaN/inf
    result = np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)
    assert len(result) == D3_DIM, f'Expected {D3_DIM}, got {len(result)}'
    return result


# 验证单条
test_vec = dict_to_vec(d_3D_raw[0])
print(f'Test vec[0]: shape={test_vec.shape}, NaNs={np.isnan(test_vec).sum()}')

# 构建全量数组
print(f'Building d_3D_arr ({len(d_3D_raw)} rows)...')
d_3D_arr = np.zeros((len(d_3D_raw), D3_DIM), dtype=np.float32)
for i in range(len(d_3D_raw)):
    d_3D_arr[i] = dict_to_vec(d_3D_raw[i])

print(f'd_3D_arr: {d_3D_arr.shape}')
nan_count = np.isnan(d_3D_arr).sum()
inf_count = np.isinf(d_3D_arr).sum()
print(f'NaN count: {nan_count}, Inf count: {inf_count}')
assert nan_count == 0, f'NaN in d_3D_arr! Fix dict_to_vec.'
assert inf_count == 0, f'Inf in d_3D_arr! Fix dict_to_vec.'
print('d_3D_arr: no NaN/Inf — OK')
del d_3D_raw
gc.collect()
print('Done.')

D3_DIM = 10+3+4+3+1+200+167+127 = 515
Test vec[0]: shape=(515,), NaNs=0
Building d_3D_arr (33062 rows)...
d_3D_arr: (33062, 515)
NaN count: 0, Inf count: 0
d_3D_arr: no NaN/Inf — OK
Done.


In [4]:
# ================================================================
# Cell 3: 采样 — 直接用行 index（行号 == pair index）
# ================================================================
MAX_PER_CLASS = 15000
SEED = 42
np.random.seed(SEED)

labels_all = df['label'].values
pos_idx    = np.where(labels_all == 1)[0]
neg_idx    = np.where(labels_all == 0)[0]

sampled_pos = np.random.choice(pos_idx, min(MAX_PER_CLASS, len(pos_idx)), replace=False)
sampled_neg = np.random.choice(neg_idx, min(MAX_PER_CLASS, len(neg_idx)), replace=False)
sampled_idx = np.concatenate([sampled_pos, sampled_neg])
np.random.shuffle(sampled_idx)
sampled_labels = labels_all[sampled_idx]

print(f'Sampled: total={len(sampled_idx)}, pos={(sampled_labels==1).sum()}, neg={(sampled_labels==0).sum()}')

Sampled: total=30000, pos=15000, neg=15000


In [5]:
# ================================================================
# Cell 4: Dataset — 蛋白质按需加载，避免全量展开 OOM
# ================================================================
MAX_PROT_LEN = 800
P_EMB_DIM    = 1024

print(f'Dims: D2={D2_DIM}, D3={D3_DIM}, P={P_EMB_DIM}, MAX_PROT_LEN={MAX_PROT_LEN}')


class DTIDataset(Dataset):
    """
    pair_indices : 原始 pair 行索引 (进入 d_2D_raw / d_3D_arr / t_feat_raw)
    labels       : 对应标签
    """
    def __init__(self, pair_indices, labels,
                 d_2D_raw, d_3D_arr, t_feat_raw,
                 max_prot_len=MAX_PROT_LEN):
        self.pair_indices = pair_indices
        self.labels       = labels
        self.d_2D         = d_2D_raw      # (N, 256) float32
        self.d_3D         = d_3D_arr      # (N, D3) float32
        self.t_feat       = t_feat_raw    # (N,) object, each (L_i, 1024)
        self.max_prot_len = max_prot_len

    def __len__(self):
        return len(self.pair_indices)

    def __getitem__(self, idx):
        i   = int(self.pair_indices[idx])
        lbl = int(self.labels[idx])

        d2 = self.d_2D[i].astype(np.float32)  # (256,)
        d3 = self.d_3D[i].astype(np.float32)  # (D3,)

        # 蛋白质: 按需读取单条，截断
        p = self.t_feat[i]
        if not isinstance(p, np.ndarray):
            p = np.array(p, dtype=np.float32)
        else:
            p = p.astype(np.float32)
        if p.ndim == 1:
            p = p.reshape(1, -1)
        p = p[:self.max_prot_len]  # (L, 1024)

        return (torch.from_numpy(d2),
                torch.from_numpy(d3),
                torch.from_numpy(p),
                lbl)


def collate_fn(batch):
    d2_list, d3_list, p_list, lbl_list = zip(*batch)
    d2   = torch.stack(d2_list, 0)                   # [B, 256]
    d3   = torch.stack(d3_list, 0)                   # [B, D3]
    lbls = torch.tensor(lbl_list, dtype=torch.long)  # [B]

    lens    = [p.shape[0] for p in p_list]
    max_len = max(lens)
    P_DIM   = p_list[0].shape[1]
    B       = len(p_list)
    p_pad   = torch.zeros(B, max_len, P_DIM)
    mask    = torch.zeros(B, max_len, dtype=torch.bool)
    for i, (p, L) in enumerate(zip(p_list, lens)):
        p_pad[i, :L] = p
        mask[i, :L]  = True
    return d2, d3, p_pad, mask, lbls


print('Dataset + collate_fn defined.')

Dims: D2=256, D3=515, P=1024, MAX_PROT_LEN=800
Dataset + collate_fn defined.


In [6]:
# ================================================================
# Cell 5: Train / Val / Test split + DataLoaders
# ================================================================
BATCH_SIZE = 128

local_idx = np.arange(len(sampled_idx))
tr_loc, tmp_loc, _, tmp_lbl = train_test_split(
    local_idx, sampled_labels, test_size=0.20,
    stratify=sampled_labels, random_state=1)
val_loc, test_loc = train_test_split(
    tmp_loc, test_size=0.50,
    stratify=sampled_labels[tmp_loc], random_state=1)

for name, loc in [('Train', tr_loc), ('Val', val_loc), ('Test', test_loc)]:
    pos_n = sampled_labels[loc].sum()
    print(f'{name:5s}: {len(loc):6d}  pos={pos_n} ({100*pos_n/len(loc):.1f}%)')


def make_loader(loc, shuffle):
    ds = DTIDataset(
        pair_indices = sampled_idx[loc],
        labels       = sampled_labels[loc],
        d_2D_raw     = d_2D_raw,
        d_3D_arr     = d_3D_arr,
        t_feat_raw   = t_feat_raw,
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True)


train_loader = make_loader(tr_loc,   shuffle=True)
val_loader   = make_loader(val_loc,  shuffle=False)
test_loader  = make_loader(test_loc, shuffle=False)
print(f'Batches — Train:{len(train_loader)} Val:{len(val_loader)} Test:{len(test_loader)}')

Train:  24000  pos=12000 (50.0%)
Val  :   3000  pos=1500 (50.0%)
Test :   3000  pos=1500 (50.0%)
Batches — Train:188 Val:24 Test:24


In [7]:
# ================================================================
# Cell 6: 模型
# ================================================================

class LightAttentionPooling(nn.Module):
    def __init__(self, in_dim, out_dim, kernel=9, dropout=0.2):
        super().__init__()
        pad = kernel // 2
        self.feat_conv = nn.Conv1d(in_dim, in_dim, kernel, padding=pad)
        self.attn_conv = nn.Conv1d(in_dim, in_dim, kernel, padding=pad)
        self.proj = nn.Sequential(
            nn.Linear(2 * in_dim, out_dim),
            nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        # x: [B, L, in_dim]  mask: [B, L] bool
        # Cast to float32 before convolutions to avoid fp16 overflow
        x    = x.float()
        xt   = x.permute(0, 2, 1)              # [B, C, L]
        feat = self.drop(self.feat_conv(xt))    # [B, C, L]
        attn = self.attn_conv(xt)               # [B, C, L]
        mask_t = mask.unsqueeze(1)              # [B, 1, L]
        # -1e4 is safe for both float16 and float32 (fp16 max ~65504)
        attn   = attn.masked_fill(~mask_t, -1e4)
        attn   = torch.softmax(attn, dim=2)
        weighted = (feat * attn).sum(dim=2)                            # [B, C]
        maxpool  = feat.masked_fill(~mask_t, -1e4).max(dim=2).values  # [B, C]
        return self.proj(torch.cat([weighted, maxpool], dim=1))        # [B, out_dim]


class DrugEncoder(nn.Module):
    def __init__(self, d2_dim, d3_dim, out_dim, dropout=0.2):
        super().__init__()
        self.d2_enc = nn.Sequential(
            nn.Linear(d2_dim, 512), nn.LayerNorm(512), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(512, out_dim), nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )
        self.d3_enc = nn.Sequential(
            nn.Linear(d3_dim, 512), nn.LayerNorm(512), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(512, out_dim), nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )
        self.fusion = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.LayerNorm(out_dim), nn.Dropout(dropout), nn.GELU()
        )

    def forward(self, d2, d3):
        return self.fusion(torch.cat([self.d2_enc(d2), self.d3_enc(d3)], dim=1))


class DTIModel(nn.Module):
    def __init__(self, d2_dim, d3_dim, p_dim=1024,
                 drug_out=256, prot_out=256, hidden=512, dropout=0.2):
        super().__init__()
        self.drug_enc  = DrugEncoder(d2_dim, d3_dim, drug_out, dropout)
        self.prot_enc  = LightAttentionPooling(p_dim, prot_out, kernel=9, dropout=dropout)

        ca_dim = 256
        self.drug_proj = nn.Linear(drug_out, ca_dim)
        self.prot_proj = nn.Linear(prot_out, ca_dim)
        self.ca_drug   = nn.MultiheadAttention(ca_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.ca_prot   = nn.MultiheadAttention(ca_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.ca_norm_d = nn.LayerNorm(ca_dim)
        self.ca_norm_p = nn.LayerNorm(ca_dim)

        self.decoder = nn.Sequential(
            nn.Linear(ca_dim * 2, hidden),
            nn.LayerNorm(hidden), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2), nn.Dropout(dropout), nn.GELU(),
            nn.Linear(hidden // 2, 2)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, d2, d3, p, mask):
        drug_h = self.drug_enc(d2, d3)            # [B, drug_out]
        prot_h = self.prot_enc(p, mask)            # [B, prot_out]

        dq = self.drug_proj(drug_h).unsqueeze(1)  # [B, 1, ca_dim]
        pk = self.prot_proj(prot_h).unsqueeze(1)  # [B, 1, ca_dim]

        d2p, _ = self.ca_drug(dq, pk, pk)         # drug ← prot context
        p2d, _ = self.ca_prot(pk, dq, dq)         # prot ← drug context

        drug_ca = self.ca_norm_d(self.drug_proj(drug_h) + d2p.squeeze(1))  # [B, ca_dim]
        prot_ca = self.ca_norm_p(self.prot_proj(prot_h) + p2d.squeeze(1))  # [B, ca_dim]

        return self.decoder(torch.cat([drug_ca, prot_ca], dim=1))  # [B, 2]


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DTIModel(
    d2_dim=D2_DIM, d3_dim=D3_DIM, p_dim=P_EMB_DIM,
    drug_out=256, prot_out=256, hidden=512, dropout=0.2
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {n_params:,}')

# Sanity check — verify inputs and model output
d2b, d3b, pb, mkb, lblb = next(iter(train_loader))
assert not d2b.isnan().any(), 'NaN in d2 batch!'
assert not d3b.isnan().any(), 'NaN in d3 batch!'
assert not pb.isnan().any(),  'NaN in protein batch!'
print(f'd2 range: [{d2b.min():.3f}, {d2b.max():.3f}]')
print(f'd3 range: [{d3b.min():.3f}, {d3b.max():.3f}]')
print(f'p  range: [{pb.min():.3f}, {pb.max():.3f}]')
with torch.no_grad():
    out = model(
        d2b.to(device).float(), d3b.to(device).float(),
        pb.to(device).float(), mkb.to(device)
    )
assert not out.isnan().any(), f'NaN in model output! Check architecture.'
prob = torch.softmax(out.float(), dim=1)[:, 1]
print(f'Output shape: {out.shape},  prob mean={prob.mean():.3f}, std={prob.std():.4f}')
assert prob.std() > 0.005, 'COLLAPSED output!'
print('Sanity check PASSED.')

Model params: 21,249,794
d2 range: [-3.571, 3.998]
d3 range: [-1.427, 62.000]
p  range: [-1.636, 1.467]
Output shape: torch.Size([128, 2]),  prob mean=0.555, std=0.2204
Sanity check PASSED.


In [8]:
# ================================================================
# Cell 7: 训练
# ================================================================
MAX_EPOCH = 30
BASE_LR   = 3e-4
WD        = 1e-4
PATIENCE  = 20
CKPT      = '/kaggle/working/best_model_v17.pt'

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr    = BASE_LR,
    total_steps = MAX_EPOCH * len(train_loader),
    pct_start   = 0.10,
    anneal_strategy = 'cos',
    div_factor      = 25.0,
    final_div_factor= 1000.0,
)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
print(f'LR={BASE_LR}, BS={BATCH_SIZE}, MaxEpoch={MAX_EPOCH}')


def run_epoch(loader, ep, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for step, (d2, d3, p, mask, lbl) in enumerate(loader):
            d2   = d2.to(device, non_blocking=True).float()
            d3   = d3.to(device, non_blocking=True).float()
            p    = p.to(device, non_blocking=True).float()
            mask = mask.to(device, non_blocking=True)
            lbl  = lbl.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            if scaler and train:
                with torch.cuda.amp.autocast():
                    logits = model(d2, d3, p, mask)
                    loss   = F.cross_entropy(logits, lbl, label_smoothing=0.05)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            else:
                logits = model(d2, d3, p, mask)
                loss   = F.cross_entropy(logits, lbl, label_smoothing=0.05)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()

            total_loss += loss.item()
            with torch.no_grad():
                prob = torch.softmax(logits.float(), dim=1)[:, 1]
                pred = logits.argmax(dim=1)
            all_preds.append(pred.cpu())
            all_labels.append(lbl.cpu())
            all_probs.append(prob.cpu())

            if train and step % 50 == 49:
                ba = (pred.cpu() == lbl.cpu()).float().mean().item() * 100
                print(f'  Ep{ep} [{step+1:3d}/{len(loader)}] '
                      f'loss={loss.item():.4f} acc={ba:.1f}% '
                      f'std={prob.cpu().std():.4f} '
                      f'LR={optimizer.param_groups[0]["lr"]:.2e}')

    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.cat(all_probs).numpy()
    avg_loss = total_loss / max(len(loader), 1)
    acc      = float((preds == labels).mean())
    return avg_loss, acc, preds, labels, probs


best_auc   = 0.0
no_improve = 0
start_time = time.time()

for ep in range(1, MAX_EPOCH + 1):
    tr_loss, tr_acc, _, _, _ = run_epoch(train_loader, ep, train=True)

    val_loss, val_acc, val_preds, val_labels, val_probs = run_epoch(
        val_loader, ep, train=False)

    try:
        val_auc  = roc_auc_score(val_labels, val_probs)
        val_aupr = average_precision_score(val_labels, val_probs)
    except Exception:
        val_auc = val_aupr = float('nan')

    val_f1  = f1_score(val_labels, val_preds, zero_division=0)
    val_mcc = matthews_corrcoef(val_labels, val_preds)
    elapsed = (time.time() - start_time) / 60

    print(f'[Ep {ep:3d}/{MAX_EPOCH}] '
          f'tr={100*tr_acc:.1f}% val={100*val_acc:.1f}% '
          f'AUC={val_auc:.3f} AUPR={val_aupr:.3f} '
          f'F1={val_f1:.3f} MCC={val_mcc:.3f} '
          f'({elapsed:.1f}min)')

    is_nan = val_auc != val_auc
    if (not is_nan) and val_auc > best_auc:
        best_auc   = val_auc
        no_improve = 0
        torch.save(model.state_dict(), CKPT)
        print(f'  ★ Best AUC={best_auc:.4f} saved.')
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'Early stop at epoch {ep}.')
        break

print(f'Done. Best val AUC={best_auc:.4f}')

LR=0.0003, BS=128, MaxEpoch=30
  Ep1 [ 50/188] loss=0.7196 acc=60.2% std=0.2044 LR=1.76e-05
  Ep1 [100/188] loss=0.6884 acc=58.6% std=0.2116 LR=3.38e-05
  Ep1 [150/188] loss=0.6092 acc=68.8% std=0.2280 LR=5.96e-05
[Ep   1/30] tr=59.7% val=70.4% AUC=0.794 AUPR=0.807 F1=0.638 MCC=0.437 (2.0min)
  ★ Best AUC=0.7944 saved.
  Ep2 [ 50/188] loss=0.4988 acc=80.5% std=0.2418 LR=1.21e-04
  Ep2 [100/188] loss=0.5801 acc=70.3% std=0.2595 LR=1.61e-04
  Ep2 [150/188] loss=0.5273 acc=74.2% std=0.2579 LR=2.01e-04
[Ep   2/30] tr=71.5% val=72.9% AUC=0.820 AUPR=0.826 F1=0.739 MCC=0.460 (3.8min)
  ★ Best AUC=0.8201 saved.
  Ep3 [ 50/188] loss=0.5371 acc=71.1% std=0.2908 LR=2.60e-04
  Ep3 [100/188] loss=0.5796 acc=73.4% std=0.2774 LR=2.83e-04
  Ep3 [150/188] loss=0.4631 acc=81.2% std=0.2624 LR=2.97e-04
[Ep   3/30] tr=75.7% val=75.3% AUC=0.849 AUPR=0.850 F1=0.728 MCC=0.515 (5.6min)
  ★ Best AUC=0.8494 saved.
  Ep4 [ 50/188] loss=0.4774 acc=80.5% std=0.2956 LR=3.00e-04
  Ep4 [100/188] loss=0.5502 acc=72.7% 

In [9]:
# ================================================================
# Cell 8: TEST 评估
# ================================================================
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=False))
model.eval()

_, te_acc, te_preds, te_labels, te_probs = run_epoch(test_loader, MAX_EPOCH, train=False)

try:
    te_auc  = roc_auc_score(te_labels, te_probs)
    te_aupr = average_precision_score(te_labels, te_probs)
except Exception:
    te_auc = te_aupr = float('nan')

te_f1   = f1_score(te_labels, te_preds, zero_division=0)
te_mcc  = matthews_corrcoef(te_labels, te_preds)
te_prec = precision_score(te_labels, te_preds, zero_division=0)
te_rec  = recall_score(te_labels, te_preds, zero_division=0)
cm      = confusion_matrix(te_labels, te_preds)

print('=' * 55)
print('TEST RESULTS')
print('=' * 55)
print(f'Acc:  {100*te_acc:.2f}%')
print(f'AUC:  {te_auc:.4f}')
print(f'AUPR: {te_aupr:.4f}')
print(f'F1:   {te_f1:.4f}')
print(f'MCC:  {te_mcc:.4f}')
print(f'Prec: {te_prec:.4f}')
print(f'Rec:  {te_rec:.4f}')
print(f'Confusion: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}')
print('=' * 55)

TEST RESULTS
Acc:  82.10%
AUC:  0.8975
AUPR: 0.8883
F1:   0.8224
MCC:  0.6421
Prec: 0.8162
Rec:  0.8287
Confusion: TN=1220 FP=280 FN=257 TP=1243
